# Loss functions

> A set of custom loss functions

In [2]:
#| default_exp losses

In [3]:
#| hide
from nbdev.showdoc import *

# Loss Functions

In [4]:
#| export
from fastai.vision.all import *
from monai.losses import SSIMLoss


### Combined Loss

In [5]:
#| export
class CombinedLoss:
    "losses combined"
    def __init__(self, spatial_dims=2, alpha=0.33, beta=0.33):
        store_attr()
        self.SSIM_loss = SSIMLoss(spatial_dims=spatial_dims)
        self.MSE_loss =  nn.MSELoss()
        self.MAE_loss =  nn.L1Loss()
        
    def __call__(self, pred, targ):
        return (1 - self.alpha - self.beta) * self.SSIM_loss(pred, targ) + self.alpha * self.MSE_loss(pred, targ) + self.beta * self.MAE_loss(pred, targ)

### Dice Loss

In [9]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1):
        super(DiceLoss, self).__init__()
        self.smooth = smooth

    def forward(self, inputs, targets):
        # Make sure the inputs are probabilities
        inputs = torch.sigmoid(inputs)

        # Flatten tensors
        inputs = inputs.view(-1)
        targets = targets.view(-1)

        # Calculate the intersection
        intersection = (inputs * targets).sum()

        # Compute Dice Coefficient
        dice = (2. * intersection + self.smooth) / (inputs.sum() + targets.sum() + self.smooth)

        # Copmute dice loss
        loss = 1 - dice

        return loss


# inputs and targets must be equally dimensional tensors

inputs = torch.randn((1, 1, 256, 256))  # Input
targets = torch.randint(0, 2, (1, 1, 256, 256)).float()  # Ground Truth


# Inicialize
dice_loss = DiceLoss()

# Compute loss
loss = dice_loss(inputs, targets)
print('Dice Loss:', loss.item())

print("Input",inputs)
print("Target",targets)

Dice Loss: 0.5002192854881287
Input tensor([[[[-0.5154,  0.7132,  0.3507,  ...,  2.0297, -0.8504,  0.9955],
          [-0.6084, -0.4531, -0.3083,  ...,  0.0067,  2.6408, -1.2425],
          [ 0.6920,  0.0522,  0.6456,  ..., -0.1864, -1.2513,  0.1199],
          ...,
          [-1.7286,  0.2760,  0.0166,  ..., -0.5510,  0.1383,  0.0180],
          [ 0.4423,  0.0688, -0.3138,  ..., -0.2872,  1.0493,  0.4146],
          [ 0.0200,  0.5943,  0.8527,  ...,  0.2318,  1.2477,  2.2307]]]])
Target tensor([[[[1., 1., 1.,  ..., 0., 1., 0.],
          [1., 1., 0.,  ..., 0., 1., 1.],
          [0., 1., 0.,  ..., 1., 1., 1.],
          ...,
          [1., 0., 1.,  ..., 0., 1., 0.],
          [1., 0., 0.,  ..., 1., 1., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]]])


# Metrics

In [7]:
#| export

def SSIM(x, y, spatial_dims=2):
    return 1 - SSIMLoss(spatial_dims)(x,y)

SSIMMetric = AvgMetric(SSIM)

In [8]:
#| hide
import nbdev; nbdev.nbdev_export()